### Classification Model for our dataset

In [307]:
import pandas as pd
import numpy as np
from configs.system import project_dir

In [308]:
p_dir = project_dir()
data_dir = p_dir.FEATURED_DIR

dataset_path = data_dir/'featured_dataset.csv'

In [309]:
df = pd.read_csv(dataset_path,index_col=0)

df.sample(2)


,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,...,Duration,Demand,StartMonth,StartWeek,EndMonth,EndWeek,DemandPerMonth,StartDayOfWeek,DemandPerDay,Shift
18303,2132,107,1,public area,0,2021-10-31,21:16:57,2021-11-01,00:18:00,2021-10-31 21:16:00,...,181,9.29,10,6,11,0,64933.48,Sunday,17.81,Night
70284,466,1971,0,public institution,1,2022-09-01,12:31:22,2022-09-01,13:11:23,2022-09-01 12:31:00,...,40,30.51,9,3,9,3,111771.80,Thursday,17.62,Day


In [310]:
df['Target_Demand'] = (df['Demand']>df['Demand'].median()).astype(int)

In [311]:
df.sample(3)

,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,...,Demand,StartMonth,StartWeek,EndMonth,EndWeek,DemandPerMonth,StartDayOfWeek,DemandPerDay,Shift,Target_Demand
60527,474,1056,0,public institution,1,2022-03-13,18:37:53,2022-03-13,19:17:57,2022-03-13 18:37:00,...,19.65,3,6,3,6,103896.05,Sunday,17.81,Night,1
24884,76,173,1,public area,0,2022-02-21,08:25:33,2022-02-21,11:37:48,2022-02-21 08:25:00,...,8.46,2,0,2,0,103141.96,Monday,17.35,Day,0
68382,258,1701,0,hotel,0,2022-05-28,16:22:29,2022-05-28,19:32:28,2022-05-28 16:22:00,...,11.60,5,5,5,5,115283.83,Saturday,17.41,Night,0


In [312]:
df.columns

Index(['UserID', 'ChargerID', 'ChargerCompany', 'Location', 'ChargerType',
       'StartDay', 'StartTime', 'EndDay', 'EndTime', 'StartDatetime',
       'EndDatetime', 'Duration', 'Demand', 'StartMonth', 'StartWeek',
       'EndMonth', 'EndWeek', 'DemandPerMonth', 'StartDayOfWeek',
       'DemandPerDay', 'Shift', 'Target_Demand'],
      dtype='str')

In [313]:
categorical_columns = ['Shift','Location']
numerical_columns = ['ChargerCompany','ChargerType','Duration','DemandPerDay']
ordinal_columns = ['StartDayOfWeek']

In [314]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder

preprocessor = ColumnTransformer([
    ('cat',OneHotEncoder(),categorical_columns),
    ('num',StandardScaler(),numerical_columns),
    ('ord',OrdinalEncoder(),ordinal_columns)
])

In [315]:
X = df[['Shift','Location','ChargerCompany','ChargerType','Duration','DemandPerDay','StartDayOfWeek']]
y = df['Target_Demand']

In [316]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [317]:
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

### Decision Tree Classifier

In [318]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

depths = [4,5,6,7,8,9,10,13,15,20]

dt_r = []

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth,random_state=42)

    dt.fit(X_train_prep,y_train)

    y_pred = dt.predict(X_test_prep)

    y_pred_train = dt.predict(X_train_prep)

    accuracy_test = accuracy_score(y_pred,y_test)

    accuracy_train = accuracy_score(y_pred_train,y_train)

    dt_r.append({
        'depth' : depth,
        'test' : accuracy_test,
        'train' : accuracy_train
    })



In [319]:
dt_r

[{'depth': 4, 'test': 0.8885074831800083, 'train': 0.8861483007209063},
 {'depth': 5, 'test': 0.8891253604283949, 'train': 0.8884998283556471},
 {'depth': 6, 'test': 0.8909103391459563, 'train': 0.8912461380020598},
 {'depth': 7, 'test': 0.8913222573115475, 'train': 0.8926879505664264},
 {'depth': 8, 'test': 0.8907043800631608, 'train': 0.8940954342602129},
 {'depth': 9, 'test': 0.8891253604283949, 'train': 0.8968589083419155},
 {'depth': 10, 'test': 0.8901551558423727, 'train': 0.9002403020940611},
 {'depth': 13, 'test': 0.8830838939997254, 'train': 0.9134225883968418},
 {'depth': 15, 'test': 0.8756693670190855, 'train': 0.9227943700652248},
 {'depth': 20, 'test': 0.8639296992997392, 'train': 0.9413491246138002}]

In [320]:
splits = [5,10,15,20,25,30]

dt_r = []

for split in splits:
    dt = DecisionTreeClassifier(min_samples_split = split,random_state=42)

    dt.fit(X_train_prep,y_train)

    y_pred = dt.predict(X_test_prep)

    y_pred_train = dt.predict(X_train_prep)

    accuracy_test = accuracy_score(y_test,y_pred)

    accuracy_train = accuracy_score(y_train,y_pred_train)

    dt_r.append({
        'split' : split,
        'test' : accuracy_test,
        'train' : accuracy_train
    })

In [321]:
dt_r

[{'split': 5, 'test': 0.8624193326925718, 'train': 0.9340370751802266},
 {'split': 10, 'test': 0.871344226280379, 'train': 0.9220906282183317},
 {'split': 15, 'test': 0.8771797336262529, 'train': 0.9156882938551322},
 {'split': 20, 'test': 0.8804750789509818, 'train': 0.9116889804325438},
 {'split': 25, 'test': 0.8819167925305506, 'train': 0.9089941640920014},
 {'split': 30, 'test': 0.8829465879445284, 'train': 0.906951596292482}]

In [322]:
from sklearn.metrics import f1_score, classification_report,roc_auc_score

In [323]:
dt = DecisionTreeClassifier(max_depth=10,min_samples_split=20,random_state=42)

dt.fit(X_train_prep,y_train)

y_pred = dt.predict(X_test_prep)

y_pred_prob = dt.predict_proba(X_test_prep)[:,1]






In [324]:
f1_s = f1_score(y_test,y_pred)
# f1_train = f1_score(y_train,y_pred_train)

roc_test = roc_auc_score(y_test,y_pred_prob)
accuracy = accuracy_score(y_test,y_pred)

report_dt = classification_report(y_test,y_pred)

results = []
results.append({
    'model' : 'Decision Tree',
    'Accuracy' : accuracy,
    'ROC' : roc_test,
    'F1' : f1_s
})

# print(results)
print(report_dt)

results


              precision    recall  f1-score   support

           0       0.91      0.87      0.89      7280
           1       0.88      0.91      0.89      7286

    accuracy                           0.89     14566
   macro avg       0.89      0.89      0.89     14566
weighted avg       0.89      0.89      0.89     14566



[{'model': 'Decision Tree',
  'Accuracy': 0.8913222573115475,
  'ROC': 0.9492963888293973,
  'F1': 0.8934222042684979}]

## Random forest Classifier

In [325]:
from sklearn.ensemble import RandomForestClassifier

In [326]:
depths = [4,5,6,7,8,9,10,11,13,15]

rf_r = []

for depth in depths:
    rf = RandomForestClassifier(max_depth=depth,random_state=42,n_jobs=-1)
    rf.fit(X_train_prep,y_train)

    y_pred = rf.predict(X_test_prep)

    y_pred_train = rf.predict(X_train_prep)

    report = classification_report(y_test,y_pred)

    # print(report)

    accuracy = accuracy_score(y_test,y_pred)
    accuracy_train = accuracy_score(y_train,y_pred_train)

    rf_r.append({
        'depth' : depth,
        'accuracy_test' : accuracy,
        'train' : accuracy_train
    })


In [327]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

dt = DecisionTreeClassifier(
    max_depth=13,
    random_state=42
)

dt.fit(X_train_prep, y_train)

y_pred = dt.predict(X_test_prep)
y_pred_prob = dt.predict_proba(X_test_prep)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_prob)
f1 = f1_score(y_test, y_pred)

results.append({
    'model': 'Decision Tree',
    'Accuracy': accuracy,
    'ROC': roc_auc,
    'F1': f1
})

In [328]:

y_pred_prob = dt.predict_proba(X_test_prep)
y_pred_prob

array([[0.45070423, 0.54929577],
       [0.02758621, 0.97241379],
       [0.        , 1.        ],
       ...,
       [1.        , 0.        ],
       [0.        , 1.        ],
       [0.99530149, 0.00469851]], shape=(14566, 2))

In [329]:
results

[{'model': 'Decision Tree',
  'Accuracy': 0.8913222573115475,
  'ROC': 0.9492963888293973,
  'F1': 0.8934222042684979},
 {'model': 'Decision Tree',
  'Accuracy': 0.8830838939997254,
  'ROC': 0.92674565175423,
  'F1': 0.885481810234685}]

In [330]:
rf_r

[{'depth': 4,
  'accuracy_test': 0.8765618563778662,
  'train': 0.8729660144181256},
 {'depth': 5, 'accuracy_test': 0.8775229987642456, 'train': 0.874133196017851},
 {'depth': 6,
  'accuracy_test': 0.8799945077577921,
  'train': 0.8769138345348438},
 {'depth': 7,
  'accuracy_test': 0.8872030756556364,
  'train': 0.8870236869207003},
 {'depth': 8,
  'accuracy_test': 0.8885761362076068,
  'train': 0.8904909028492962},
 {'depth': 9,
  'accuracy_test': 0.8926953178635177,
  'train': 0.8955544112598696},
 {'depth': 10,
  'accuracy_test': 0.8929699299739119,
  'train': 0.8982835564709921},
 {'depth': 11,
  'accuracy_test': 0.8935878072222985,
  'train': 0.9014418125643666},
 {'depth': 13,
  'accuracy_test': 0.8964025813538377,
  'train': 0.9088225197391006},
 {'depth': 15,
  'accuracy_test': 0.8923520527255252,
  'train': 0.9196876072777206}]

In [331]:
estimators = [100,200,350,500]

rf_r = []

for n in estimators:
    rf = RandomForestClassifier(n_estimators=n,n_jobs=-1,random_state=42)

    rf.fit(X_train_prep,y_train)

    y_pred = rf.predict(X_test_prep)

    y_pred_train = rf.predict(X_train_prep)

    report = classification_report(y_test,y_pred)

    # print(report)

    accuracy = accuracy_score(y_test,y_pred)
    accuracy_train = accuracy_score(y_train,y_pred_train)

    rf_r.append({
        'estimators' : n,
        'accuracy_test' : accuracy,
        'train' : accuracy_train
    })


In [332]:
rf_r

[{'estimators': 100,
  'accuracy_test': 0.8720994095839627,
  'train': 0.9476656368005493},
 {'estimators': 200,
  'accuracy_test': 0.873472470135933,
  'train': 0.9476828012358394},
 {'estimators': 350,
  'accuracy_test': 0.8731292049979404,
  'train': 0.9476828012358394},
 {'estimators': 500,
  'accuracy_test': 0.8727859398599478,
  'train': 0.9476828012358394}]

In [333]:
splits = [5,10,15,18,20,25]

rf_r = []

for split in splits:
    rf = RandomForestClassifier(n_estimators=100,min_samples_split=split,n_jobs=-1,random_state=42)


    rf.fit(X_train_prep,y_train)
    
    y_pred = rf.predict(X_test_prep)

    y_pred_train = rf.predict(X_train_prep)

    report = classification_report(y_test,y_pred)

    accuracy = accuracy_score(y_test,y_pred)
    accuracy_train = accuracy_score(y_train,y_pred_train)

    rf_r.append({
            'split' : split,
            'accuracy_test' : accuracy,
            'train' : accuracy_train
        })
    


In [334]:
rf_r

[{'split': 5,
  'accuracy_test': 0.8764245503226692,
  'train': 0.9350326124270512},
 {'split': 10,
  'accuracy_test': 0.8850061787724839,
  'train': 0.9217130106419499},
 {'split': 15,
  'accuracy_test': 0.8881642180420156,
  'train': 0.9149158942670786},
 {'split': 18,
  'accuracy_test': 0.8901551558423727,
  'train': 0.9116718159972537},
 {'split': 20,
  'accuracy_test': 0.8909789921735548,
  'train': 0.9095090971507037},
 {'split': 25, 'accuracy_test': 0.89166552244954, 'train': 0.9069344318571919}]

In [335]:
rf = RandomForestClassifier(n_estimators=100,n_jobs=-1,random_state=42,min_samples_split=20,max_depth=13)

rf.fit(X_train_prep,y_train)

y_pred = rf.predict(X_test_prep)
y_pred_prob = rf.predict_proba(X_test_prep)[:,1]

accuracy_s = accuracy_score(y_test,y_pred)

roc_auc_s = roc_auc_score(y_test,y_pred_prob)

f1_s = f1_score(y_test,y_pred)

results.append({
    'model' : 'RandomForestClassifier',
    'Accuracy' : accuracy_s,
    'ROC' : roc_auc_s,
    'F1': f1_s
})

In [336]:
results

[{'model': 'Decision Tree',
  'Accuracy': 0.8913222573115475,
  'ROC': 0.9492963888293973,
  'F1': 0.8934222042684979},
 {'model': 'Decision Tree',
  'Accuracy': 0.8830838939997254,
  'ROC': 0.92674565175423,
  'F1': 0.885481810234685},
 {'model': 'RandomForestClassifier',
  'Accuracy': 0.8950295208018674,
  'ROC': 0.9544189537816014,
  'F1': 0.8991491326429655}]

In [337]:
results

[{'model': 'Decision Tree',
  'Accuracy': 0.8913222573115475,
  'ROC': 0.9492963888293973,
  'F1': 0.8934222042684979},
 {'model': 'Decision Tree',
  'Accuracy': 0.8830838939997254,
  'ROC': 0.92674565175423,
  'F1': 0.885481810234685},
 {'model': 'RandomForestClassifier',
  'Accuracy': 0.8950295208018674,
  'ROC': 0.9544189537816014,
  'F1': 0.8991491326429655}]

In [338]:
from sklearn.linear_model import LogisticRegression

cs =[0.01,0.1,1,10]
rf_r = []
for c in cs:
    lg = LogisticRegression(C=c,random_state=42)

    lg.fit(X_train_prep,y_train)
        
    y_pred = lg.predict(X_test_prep)

    y_pred_train = lg.predict(X_train_prep)

    report = classification_report(y_test,y_pred)

    accuracy = accuracy_score(y_test,y_pred)
    accuracy_train = accuracy_score(y_train,y_pred_train)

    rf_r.append({
            'c' : c,
            'accuracy_test' : accuracy,
            'train' : accuracy_train
        })

In [339]:
rf_r

[{'c': 0.01, 'accuracy_test': 0.8379102018399012, 'train': 0.840010298661174},
 {'c': 0.1, 'accuracy_test': 0.8400384456954552, 'train': 0.8414521112255406},
 {'c': 1, 'accuracy_test': 0.8398324866126596, 'train': 0.8412118091314795},
 {'c': 10, 'accuracy_test': 0.8402444047782507, 'train': 0.8414177823549606}]

In [340]:
max_iters = [50,100,150,200]

rf_r = []
for iter in max_iters:
    lg = LogisticRegression(max_iter=iter,random_state=42)
    
    lg.fit(X_train_prep,y_train)
            
    y_pred = lg.predict(X_test_prep)

    y_pred_train = lg.predict(X_train_prep)

    report = classification_report(y_test,y_pred)

    accuracy = accuracy_score(y_test,y_pred)
    accuracy_train = accuracy_score(y_train,y_pred_train)

    rf_r.append({
            'iter' : iter,
            'accuracy_test' : accuracy,
            'train' : accuracy_train
        })

    

/home/arson/.cache/pypoetry/virtualenvs/ev-project-8jc2ThtM-py3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 50 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=50).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [341]:
rf_r

[{'iter': 50,
  'accuracy_test': 0.8398324866126596,
  'train': 0.8412118091314795},
 {'iter': 100,
  'accuracy_test': 0.8398324866126596,
  'train': 0.8412118091314795},
 {'iter': 150,
  'accuracy_test': 0.8398324866126596,
  'train': 0.8412118091314795},
 {'iter': 200,
  'accuracy_test': 0.8398324866126596,
  'train': 0.8412118091314795}]

In [342]:
lg = LogisticRegression(max_iter=100,C=0.1,random_state=42)

lg.fit(X_train_prep,y_train)

y_pred = lg.predict(X_test_prep)
y_pred_prob = lg.predict_proba(X_test_prep)[:,1]

accuracy_s = accuracy_score(y_test,y_pred)

roc_auc_s = roc_auc_score(y_test,y_pred_prob)

f1_s = f1_score(y_test,y_pred)

results.append({
    'model' : 'LogisticRegression',
    'Accuracy' : accuracy_s,
    'ROC' : roc_auc_s,
    'F1': f1_s
})

In [343]:
results

[{'model': 'Decision Tree',
  'Accuracy': 0.8913222573115475,
  'ROC': 0.9492963888293973,
  'F1': 0.8934222042684979},
 {'model': 'Decision Tree',
  'Accuracy': 0.8830838939997254,
  'ROC': 0.92674565175423,
  'F1': 0.885481810234685},
 {'model': 'RandomForestClassifier',
  'Accuracy': 0.8950295208018674,
  'ROC': 0.9544189537816014,
  'F1': 0.8991491326429655},
 {'model': 'LogisticRegression',
  'Accuracy': 0.8400384456954552,
  'ROC': 0.9125560592646442,
  'F1': 0.8438337801608579}]

## XG Classifier

In [344]:
from xgboost import XGBClassifier

rates = [0.01,0.1,0.3,1.0]

rf_r = []

for rate in rates:
    xg = XGBClassifier(learning_rate=rate,random_state=42,n_jobs=-1)


    xg.fit(X_train_prep,y_train)
                
    y_pred = xg.predict(X_test_prep)

    y_pred_train = xg.predict(X_train_prep)

    # report = classification_report(y_test,y_pred)

    accuracy = accuracy_score(y_test,y_pred)
    accuracy_train = accuracy_score(y_train,y_pred_train)

    rf_r.append({
            'rate' : rate,
            'accuracy_test' : accuracy,
            'train' : accuracy_train
        })
    

In [345]:
rf_r

[{'rate': 0.01,
  'accuracy_test': 0.8928326239187148,
  'train': 0.891967044284243},
 {'rate': 0.1,
  'accuracy_test': 0.8948922147466704,
  'train': 0.8969790593889461},
 {'rate': 0.3,
  'accuracy_test': 0.8942056844706852,
  'train': 0.9030037761757638},
 {'rate': 1.0,
  'accuracy_test': 0.8880955650144171,
  'train': 0.9136457260556128}]

In [346]:
depths = [4,5,6,7,8,9,10,12,13,15]

rf_r = []

for depth in depths:
    xg = XGBClassifier(learning_rate=0.1,random_state=42,n_jobs=-1,max_depth=depth)


    xg.fit(X_train_prep,y_train)
                
    y_pred = xg.predict(X_test_prep)

    y_pred_train = xg.predict(X_train_prep)
    accuracy = accuracy_score(y_test,y_pred)
    accuracy_train = accuracy_score(y_train,y_pred_train)

    rf_r.append({
            'depth' : depth,
            'accuracy_test' : accuracy,
            'train' : accuracy_train
        })
    

In [347]:
rf_r

[{'depth': 4, 'accuracy_test': 0.891940134559934, 'train': 0.8907826982492276},
 {'depth': 5,
  'accuracy_test': 0.8938624193326926,
  'train': 0.8935461723309304},
 {'depth': 6,
  'accuracy_test': 0.8948922147466704,
  'train': 0.8969790593889461},
 {'depth': 7,
  'accuracy_test': 0.8951668268570644,
  'train': 0.9000171644352901},
 {'depth': 8,
  'accuracy_test': 0.8945489496086777,
  'train': 0.9020254033642293},
 {'depth': 9,
  'accuracy_test': 0.8944802965810792,
  'train': 0.9049433573635427},
 {'depth': 10,
  'accuracy_test': 0.8929012769463134,
  'train': 0.909148644009612},
 {'depth': 12,
  'accuracy_test': 0.891253604283949,
  'train': 0.9167868177136972},
 {'depth': 13,
  'accuracy_test': 0.8883015240972126,
  'train': 0.9210264332303467},
 {'depth': 15,
  'accuracy_test': 0.8850061787724839,
  'train': 0.9270683144524545}]

In [348]:
estimators = [100,150,200,250]

rf_r = []

for n in estimators:
    xg = XGBClassifier(learning_rate=0.1,random_state=42,n_jobs=-1,max_depth=7,n_estimators=n)


    xg.fit(X_train_prep,y_train)
                
    y_pred = xg.predict(X_test_prep)

    y_pred_train = xg.predict(X_train_prep)
    accuracy = accuracy_score(y_test,y_pred)
    accuracy_train = accuracy_score(y_train,y_pred_train)

    rf_r.append({
            'estimators' : n,
            'accuracy_test' : accuracy,
            'train' : accuracy_train
        })
    

In [349]:
rf_r

[{'estimators': 100,
  'accuracy_test': 0.8951668268570644,
  'train': 0.9000171644352901},
 {'estimators': 150,
  'accuracy_test': 0.8957847041054511,
  'train': 0.9018022657054583},
 {'estimators': 200,
  'accuracy_test': 0.8956473980502541,
  'train': 0.9036903535873669},
 {'estimators': 250,
  'accuracy_test': 0.8942056844706852,
  'train': 0.9050978372811535}]

In [350]:
### XG boost classifier

xg = XGBClassifier(max_depth=7,learning_rate=0.1,n_estimators=100,subsample=0.80,colsample_bytree=0.80,random_state=42,n_jobs=-1)

xg.fit(X_train_prep,y_train)

y_pred = xg.predict(X_test_prep)
y_pred_prob = xg.predict_proba(X_test_prep)[:,1]

accuracy = accuracy_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)

roc = roc_auc_score(y_test,y_pred_prob)


results.append({
    'model' : 'XGBClassifier',
    'Accuracy' : accuracy,
    'ROC' : roc,
    'F1' : f1
})


In [351]:
results

[{'model': 'Decision Tree',
  'Accuracy': 0.8913222573115475,
  'ROC': 0.9492963888293973,
  'F1': 0.8934222042684979},
 {'model': 'Decision Tree',
  'Accuracy': 0.8830838939997254,
  'ROC': 0.92674565175423,
  'F1': 0.885481810234685},
 {'model': 'RandomForestClassifier',
  'Accuracy': 0.8950295208018674,
  'ROC': 0.9544189537816014,
  'F1': 0.8991491326429655},
 {'model': 'LogisticRegression',
  'Accuracy': 0.8400384456954552,
  'ROC': 0.9125560592646442,
  'F1': 0.8438337801608579},
 {'model': 'XGBClassifier',
  'Accuracy': 0.8948235617190718,
  'ROC': 0.9571053114809976,
  'F1': 0.8984488930133899}]

### Gradient Boosting

In [352]:
from sklearn.ensemble import GradientBoostingClassifier

In [353]:


rates = [0.01,0.1,0.3,1.0]

rf_r = []

for rate in rates:
    gb = GradientBoostingClassifier(learning_rate=rate,random_state=42)
    gb.fit(X_train_prep,y_train)
                
    y_pred = gb.predict(X_test_prep)

    y_pred_train = gb.predict(X_train_prep)


    accuracy = accuracy_score(y_test,y_pred)

    accuracy_train = accuracy_score(y_train,y_pred_train)

    rf_r.append({
            'rate' : rate,
            'accuracy_test' : accuracy,
            'train' : accuracy_train
        })
    

In [354]:
rf_r

[{'rate': 0.01,
  'accuracy_test': 0.8839763833585061,
  'train': 0.8832303467215928},
 {'rate': 0.1,
  'accuracy_test': 0.8911162982287519,
  'train': 0.8889632681084793},
 {'rate': 0.3,
  'accuracy_test': 0.8942056844706852,
  'train': 0.8946790250600756},
 {'rate': 1.0,
  'accuracy_test': 0.8900865028147741,
  'train': 0.8928939237899073}]

In [355]:
estimators = [100,150,200,250]

rf_r = []

for n in estimators:
    gb = GradientBoostingClassifier(learning_rate=0.1,random_state=42,n_estimators=n)


    gb.fit(X_train_prep,y_train)
                
    y_pred = gb.predict(X_test_prep)

    y_pred_train = gb.predict(X_train_prep)
    accuracy = accuracy_score(y_test,y_pred)
    accuracy_train = accuracy_score(y_train,y_pred_train)

    rf_r.append({
            'estimators' : n,
            'accuracy_test' : accuracy,
            'train' : accuracy_train
        })
    

In [356]:
rf_r

[{'estimators': 100,
  'accuracy_test': 0.8911162982287519,
  'train': 0.8889632681084793},
 {'estimators': 150,
  'accuracy_test': 0.891253604283949,
  'train': 0.8896841743906625},
 {'estimators': 200,
  'accuracy_test': 0.8917341754771385,
  'train': 0.8905423961551665},
 {'estimators': 250,
  'accuracy_test': 0.8925580118083207,
  'train': 0.8923446618606248}]

In [357]:
depths = [4,5,6,7,8,9,10,12,13,15]

rf_r = []

for depth in depths:
    gb = GradientBoostingClassifier(learning_rate=0.1,random_state=42,max_depth=depth,n_estimators=150)


    gb.fit(X_train_prep,y_train)
                
    y_pred = gb.predict(X_test_prep)

    y_pred_train = gb.predict(X_train_prep)
    accuracy = accuracy_score(y_test,y_pred)
    accuracy_train = accuracy_score(y_train,y_pred_train)

    rf_r.append({
            'depth' : depth,
            'accuracy_test' : accuracy,
            'train' : accuracy_train
        })
    

In [358]:
rf_r

[{'depth': 4, 'accuracy_test': 0.8922833996979267, 'train': 0.894129763130793},
 {'depth': 5,
  'accuracy_test': 0.8937251132774955,
  'train': 0.8981634054239616},
 {'depth': 6,
  'accuracy_test': 0.8947549086914733,
  'train': 0.9030552694816341},
 {'depth': 7,
  'accuracy_test': 0.8937251132774955,
  'train': 0.9080844490216272},
 {'depth': 8,
  'accuracy_test': 0.8924207057531237,
  'train': 0.9157397871610023},
 {'depth': 9,
  'accuracy_test': 0.8909789921735548,
  'train': 0.9252832131822863},
 {'depth': 10,
  'accuracy_test': 0.8891253604283949,
  'train': 0.9335049776862341},
 {'depth': 12,
  'accuracy_test': 0.8807496910613758,
  'train': 0.9455372468245795},
 {'depth': 13,
  'accuracy_test': 0.879445283537004,
  'train': 0.9474081702711981},
 {'depth': 15,
  'accuracy_test': 0.8758066730742826,
  'train': 0.9476828012358394}]

In [359]:

gb = GradientBoostingClassifier(max_depth=6,learning_rate=0.3,n_estimators=150,subsample=0.80,random_state=42)

gb.fit(X_train_prep,y_train)

y_pred = gb.predict(X_test_prep)
y_pred_prob = gb.predict_proba(X_test_prep)[:,1]

accuracy = accuracy_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)

roc = roc_auc_score(y_test,y_pred_prob)


results.append({
    'model' : 'gradient Boosting',
    'Accuracy' : accuracy,
    'ROC' : roc,
    'F1' : f1
})


In [360]:
results

[{'model': 'Decision Tree',
  'Accuracy': 0.8913222573115475,
  'ROC': 0.9492963888293973,
  'F1': 0.8934222042684979},
 {'model': 'Decision Tree',
  'Accuracy': 0.8830838939997254,
  'ROC': 0.92674565175423,
  'F1': 0.885481810234685},
 {'model': 'RandomForestClassifier',
  'Accuracy': 0.8950295208018674,
  'ROC': 0.9544189537816014,
  'F1': 0.8991491326429655},
 {'model': 'LogisticRegression',
  'Accuracy': 0.8400384456954552,
  'ROC': 0.9125560592646442,
  'F1': 0.8438337801608579},
 {'model': 'XGBClassifier',
  'Accuracy': 0.8948235617190718,
  'ROC': 0.9571053114809976,
  'F1': 0.8984488930133899},
 {'model': 'gradient Boosting',
  'Accuracy': 0.8876836468488261,
  'ROC': 0.9521523288679479,
  'F1': 0.8910059960026648}]

## Light GBM classifier

In [361]:
from lightgbm import LGBMClassifier

In [362]:
leaves = [7,8,9,10,12,13,15,20,24,28]

rf_r = []

for leave in leaves:
    lgm = LGBMClassifier(learning_rate=0.1,random_state=42,num_leaves=leave,n_jobs=-1)


    lgm.fit(X_train_prep,y_train)
                
    y_pred = lgm.predict(X_test_prep)

    y_pred_train = lgm.predict(X_train_prep)
    accuracy = accuracy_score(y_test,y_pred)
    accuracy_train = accuracy_score(y_train,y_pred_train)

    rf_r.append({
            'leave' : leave,
            'accuracy_test' : accuracy,
            'train' : accuracy_train
        })
    

[LightGBM] [Info] Number of positive: 29095, number of negative: 29165
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.039162 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 306
[LightGBM] [Info] Number of data points in the train set: 58260, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499399 -> initscore=-0.002403
[LightGBM] [Info] Start training from score -0.002403
[LightGBM] [Info] Number of positive: 29095, number of negative: 29165
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024742 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 306
[LightGBM] [Info] Number of data points in the train set: 58260, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499399 -> initscore=-0.002403
[LightGBM] [Info] Start training from score -0.002403
[LightGBM] [In

In [363]:
rf_r

[{'leave': 7,
  'accuracy_test': 0.8914595633667445,
  'train': 0.8901132852729146},
 {'leave': 8,
  'accuracy_test': 0.8909789921735548,
  'train': 0.8900961208376245},
 {'leave': 9,
  'accuracy_test': 0.8922147466703282,
  'train': 0.8905938894610367},
 {'leave': 10,
  'accuracy_test': 0.8929699299739119,
  'train': 0.8909543426021284},
 {'leave': 12,
  'accuracy_test': 0.8924207057531237,
  'train': 0.8909715070374185},
 {'leave': 13,
  'accuracy_test': 0.8922833996979267,
  'train': 0.8912289735667697},
 {'leave': 15,
  'accuracy_test': 0.8932445420843059,
  'train': 0.892378990731205},
 {'leave': 20,
  'accuracy_test': 0.8932445420843059,
  'train': 0.8937521455544113},
 {'leave': 24,
  'accuracy_test': 0.8947549086914733,
  'train': 0.8942327497425334},
 {'leave': 28,
  'accuracy_test': 0.8950295208018674,
  'train': 0.8949708204600069}]

In [364]:
depths = [4,5,6,7,8,9,10,12,13,15]

rf_r = []

for depth in depths:
    lgm = LGBMClassifier(learning_rate=0.1,random_state=42,max_depth=depth,n_estimators=150)


    lgm.fit(X_train_prep,y_train)
                
    y_pred = lgm.predict(X_test_prep)

    y_pred_train = lgm.predict(X_train_prep)
    accuracy = accuracy_score(y_test,y_pred)
    accuracy_train = accuracy_score(y_train,y_pred_train)

    rf_r.append({
            'depth' : depth,
            'accuracy_test' : accuracy,
            'train' : accuracy_train
        })
    

[LightGBM] [Info] Number of positive: 29095, number of negative: 29165
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000644 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 306
[LightGBM] [Info] Number of data points in the train set: 58260, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499399 -> initscore=-0.002403
[LightGBM] [Info] Start training from score -0.002403
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

In [365]:
rf_r

[{'depth': 4,
  'accuracy_test': 0.8945489496086777,
  'train': 0.8931857191898387},
 {'depth': 5,
  'accuracy_test': 0.8939997253878896,
  'train': 0.8966014418125644},
 {'depth': 6,
  'accuracy_test': 0.8943429905258822,
  'train': 0.8976313079299691},
 {'depth': 7,
  'accuracy_test': 0.8944802965810792,
  'train': 0.8983007209062822},
 {'depth': 8,
  'accuracy_test': 0.8948235617190718,
  'train': 0.8980775832475112},
 {'depth': 9,
  'accuracy_test': 0.8937937663050941,
  'train': 0.8975626501888088},
 {'depth': 10,
  'accuracy_test': 0.8939310723602911,
  'train': 0.8983350497768623},
 {'depth': 12,
  'accuracy_test': 0.8948235617190718,
  'train': 0.8981119121180913},
 {'depth': 13,
  'accuracy_test': 0.8953041329122614,
  'train': 0.8982492276004119},
 {'depth': 15,
  'accuracy_test': 0.8949608677742689,
  'train': 0.8985753518709234}]

In [366]:

lgm = LGBMClassifier(max_depth=8,learning_rate=0.1,n_estimators=150,subsample=0.80,random_state=42,num_leaves=20)

lgm.fit(X_train_prep,y_train)

y_pred = lgm.predict(X_test_prep)
y_pred_prob = lgm.predict_proba(X_test_prep)[:,1]

accuracy = accuracy_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)

roc = roc_auc_score(y_test,y_pred_prob)


results.append({
    'model' : 'Light GMB',
    'Accuracy' : accuracy,
    'ROC' : roc,
    'F1' : f1
})


[LightGBM] [Info] Number of positive: 29095, number of negative: 29165
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000794 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 306
[LightGBM] [Info] Number of data points in the train set: 58260, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499399 -> initscore=-0.002403
[LightGBM] [Info] Start training from score -0.002403


In [367]:
results

[{'model': 'Decision Tree',
  'Accuracy': 0.8913222573115475,
  'ROC': 0.9492963888293973,
  'F1': 0.8934222042684979},
 {'model': 'Decision Tree',
  'Accuracy': 0.8830838939997254,
  'ROC': 0.92674565175423,
  'F1': 0.885481810234685},
 {'model': 'RandomForestClassifier',
  'Accuracy': 0.8950295208018674,
  'ROC': 0.9544189537816014,
  'F1': 0.8991491326429655},
 {'model': 'LogisticRegression',
  'Accuracy': 0.8400384456954552,
  'ROC': 0.9125560592646442,
  'F1': 0.8438337801608579},
 {'model': 'XGBClassifier',
  'Accuracy': 0.8948235617190718,
  'ROC': 0.9571053114809976,
  'F1': 0.8984488930133899},
 {'model': 'gradient Boosting',
  'Accuracy': 0.8876836468488261,
  'ROC': 0.9521523288679479,
  'F1': 0.8910059960026648},
 {'model': 'Light GMB',
  'Accuracy': 0.893656460249897,
  'ROC': 0.9568252602462044,
  'F1': 0.8969874310035246}]

In [373]:
import pickle
model_dir = p_dir.MODEL_DIR
model_dir

with open (model_dir/'classification_rf','wb') as file:
    pickle.dump(rf,file)